In [1]:
!pip install catboost

# CATBoost

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('final_input.csv')
X = df.drop(['AB_used', 'sample-name'], axis=1)
y = df['AB_used']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
mean_value = X_train.mean().mean()


imputer = SimpleImputer(strategy='constant', fill_value=mean_value)
pipeline = Pipeline([
    ('imputer', imputer), 
    ('catboost', CatBoostClassifier(silent=True)) 
])


param_grid = {
    'catboost__iterations': [100, 200],  
    'catboost__depth': [3, 5, 7],  
    'catboost__learning_rate': [0.01, 0.001]  
}


grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', verbose=1, error_score='raise')
grid_search.fit(X_train, y_train)


print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)
y_pred = grid_search.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Improved Accuracy: {accuracy}')

y_pred_proba = grid_search.predict_proba(X_test)[:, 1]


auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)
y_pred = grid_search.predict(X_test)


tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

print("Specificity:", specificity)
print("Sensitivity:", sensitivity)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters: {'catboost__depth': 7, 'catboost__iterations': 200, 'catboost__learning_rate': 0.01}
Best Score: 0.860496413381578
Improved Accuracy: 0.8740157480314961
AUC: 0.955196636982134
Specificity: 0.88268156424581
Sensitivity: 0.8663366336633663


In [2]:
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Precision: 0.8928571428571429
Recall: 0.8663366336633663
F1 Score: 0.8793969849246231


In [3]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

bootstrap_aucs = []
bootstrap_accuracies = []
bootstrap_precisions = []
bootstrap_recalls = []
bootstrap_f1s = []
bootstrap_specificities = []
n_bootstraps = 1000

for _ in range(n_bootstraps):
    indices = resample(np.arange(len(y_test)), replace=True)
    y_test_sample = y_test.iloc[indices]
    X_test_sample = X_test.iloc[indices]
    
    
    y_pred_sample = grid_search.predict(X_test_sample)
    y_pred_proba_sample = grid_search.predict_proba(X_test_sample)[:, 1]
    

    bootstrap_auc = roc_auc_score(y_test_sample, y_pred_proba_sample)
    bootstrap_accuracy = accuracy_score(y_test_sample, y_pred_sample)
    bootstrap_precision = precision_score(y_test_sample, y_pred_sample)
    bootstrap_recall = recall_score(y_test_sample, y_pred_sample)
    bootstrap_f1 = f1_score(y_test_sample, y_pred_sample)
    
    tn, fp, fn, tp = confusion_matrix(y_test_sample, y_pred_sample).ravel()
    bootstrap_specificity = tn / (tn + fp)
    
    bootstrap_aucs.append(bootstrap_auc)
    bootstrap_accuracies.append(bootstrap_accuracy)
    bootstrap_precisions.append(bootstrap_precision)
    bootstrap_recalls.append(bootstrap_recall)
    bootstrap_f1s.append(bootstrap_f1)
    bootstrap_specificities.append(bootstrap_specificity)


def calculate_ci(metric_list):
    ci_lower = np.percentile(metric_list, 2.5)
    ci_upper = np.percentile(metric_list, 97.5)
    return ci_lower, ci_upper

auc_ci = calculate_ci(bootstrap_aucs)
accuracy_ci = calculate_ci(bootstrap_accuracies)
precision_ci = calculate_ci(bootstrap_precisions)
recall_ci = calculate_ci(bootstrap_recalls)
f1_ci = calculate_ci(bootstrap_f1s)
specificity_ci = calculate_ci(bootstrap_specificities)

print(f"Bootstrap 95% CI for AUC: {auc_ci[0]:.3f} to {auc_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Accuracy: {accuracy_ci[0]:.3f} to {accuracy_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Precision: {precision_ci[0]:.3f} to {precision_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Recall (Sensitivity): {recall_ci[0]:.3f} to {recall_ci[1]:.3f}")
print(f"Bootstrap 95% CI for F1 Score: {f1_ci[0]:.3f} to {f1_ci[1]:.3f}")
print(f"Bootstrap 95% CI for Specificity: {specificity_ci[0]:.3f} to {specificity_ci[1]:.3f}")


Bootstrap 95% CI for AUC: 0.935 to 0.971
Bootstrap 95% CI for Accuracy: 0.840 to 0.906
Bootstrap 95% CI for Precision: 0.844 to 0.931
Bootstrap 95% CI for Recall (Sensitivity): 0.820 to 0.909
Bootstrap 95% CI for F1 Score: 0.845 to 0.910
Bootstrap 95% CI for Specificity: 0.832 to 0.927


In [4]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score


bootstrap_aucs = []
n_bootstraps = 1000

for _ in range(n_bootstraps):
    indices = resample(np.arange(len(y_test)), replace=True)
    y_test_sample = y_test.iloc[indices]
    X_test_sample = X_test.iloc[indices]
    
    y_pred_proba_sample = grid_search.predict_proba(X_test_sample)[:, 1]
    bootstrap_auc = roc_auc_score(y_test_sample, y_pred_proba_sample)
    bootstrap_aucs.append(bootstrap_auc)

ci_lower = np.percentile(bootstrap_aucs, 2.5)
ci_upper = np.percentile(bootstrap_aucs, 97.5)

print(f"Bootstrap 95% CI for AUC: {ci_lower:.3f} to {ci_upper:.3f}")

original_auc = roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1])
n_permutations = 10000
perm_aucs = []

for _ in range(n_permutations):
    y_test_permuted = np.random.permutation(y_test)
    perm_auc = roc_auc_score(y_test_permuted, grid_search.predict_proba(X_test)[:, 1])
    perm_aucs.append(perm_auc)

perm_aucs = np.array(perm_aucs)
p_value = np.mean(perm_aucs >= original_auc)

print(f"Permutation test p-value for AUC: {p_value:.3f}")


Bootstrap 95% CI for AUC: 0.938 to 0.972
Permutation test p-value for AUC: 0.000
